# Karpathy Quant Auto-Research — Experiment Analysis

Analysis of autonomous strategy-search results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 6 columns: commit, oos_sharpe, max_dd, turnover, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["oos_sharpe"] = pd.to_numeric(df["oos_sharpe"], errors="coerce")
df["max_dd"] = pd.to_numeric(df["max_dd"], errors="coerce")
df["turnover"] = pd.to_numeric(df["turnover"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    sh = row["oos_sharpe"]
    desc = row["description"]
    print(f"  #{i:3d}  sharpe={sh:.4f}  max_dd={row['max_dd']:.3f}  turnover={row['turnover']:.2f}  {desc}")

## OOS Sharpe Over Time

Track how the best (kept) `oos_sharpe` evolves as experiments progress. The running maximum shows the frontier — the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
baseline_sharpe = valid.loc[0, "oos_sharpe"]

# Show points at or above (baseline - 0.2) — the interesting region
above = valid[valid["oos_sharpe"] >= baseline_sharpe - 0.2]

# Discarded as faint background
disc = above[above["status"] == "DISCARD"]
ax.scatter(disc.index, disc["oos_sharpe"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Kept as prominent green
kept_v = above[above["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["oos_sharpe"],
           c="#2ecc71", s=50, zorder=4, label="Kept",
           edgecolors="black", linewidths=0.5)

# Running maximum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_sh = valid.loc[kept_mask, "oos_sharpe"]
running_max = kept_sh.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Annotate each kept experiment
for idx, sh in zip(kept_idx, kept_sh):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, sh),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("OOS Sharpe (higher is better)", fontsize=12)
ax.set_title(f"Quant Auto-Research Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below baseline to just above best
best_sharpe = kept_sh.max() if len(kept_sh) else baseline_sharpe
margin = max((best_sharpe - baseline_sharpe), 0.1) * 0.15
ax.set_ylim(baseline_sharpe - 0.2, best_sharpe + margin + 0.1)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline_sharpe = df.iloc[0]["oos_sharpe"]
best_sharpe = kept["oos_sharpe"].max()
best_row = kept.loc[kept["oos_sharpe"].idxmax()]

print(f"Baseline OOS Sharpe: {baseline_sharpe:.4f}")
print(f"Best OOS Sharpe:     {best_sharpe:.4f}")
print(f"Total improvement:   {best_sharpe - baseline_sharpe:+.4f}")
print(f"Best experiment:     {best_row['description']}")
print(f"Best max_dd:         {best_row['max_dd']:.3f}")
print(f"Best turnover:       {best_row['turnover']:.2f}")
print()

print("Cumulative progress per kept experiment:")
kept_sorted = kept.reset_index()
for _, row in kept_sorted.iterrows():
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: sharpe={row['oos_sharpe']:.4f}  max_dd={row['max_dd']:.3f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment
kept = df[df["status"] == "KEEP"].copy()
kept["prev_sharpe"] = kept["oos_sharpe"].shift(1)
kept["delta"] = kept["oos_sharpe"] - kept["prev_sharpe"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'Sharpe':>8}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.4f}  {row['oos_sharpe']:8.4f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.4f}  {'':>8}  TOTAL improvement over baseline")

## Risk/Reward Scatter

Plot every finished run (no crashes) as (max_dd, oos_sharpe). Hard-constraint band at max_dd = 0.35 is drawn. Clusters near the constraint edge are often overfits — treat with suspicion.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

valid = df[df["status"] != "CRASH"].dropna(subset=["max_dd", "oos_sharpe"])
color_map = {"KEEP": "#2ecc71", "DISCARD": "#cccccc"}
for status, group in valid.groupby("status"):
    ax.scatter(group["max_dd"], group["oos_sharpe"],
               c=color_map.get(status, "#888888"),
               s=30 if status == "KEEP" else 15,
               alpha=0.9 if status == "KEEP" else 0.5,
               edgecolors="black" if status == "KEEP" else "none",
               linewidths=0.5,
               label=status.title())

ax.axvline(0.35, color="#c0392b", linestyle="--", linewidth=1.5, alpha=0.7, label="Max-DD hard constraint")
ax.set_xlabel("Max drawdown (OOS)", fontsize=12)
ax.set_ylabel("OOS Sharpe", fontsize=12)
ax.set_title("Risk/Reward: OOS Sharpe vs Max Drawdown", fontsize=13)
ax.grid(True, alpha=0.2)
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

## Turnover Distribution

Annualized one-sided turnover across all finished runs. Hard-constraint line at 50 is drawn. Very-high-turnover strategies eat their edge in costs even after the cost model.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
valid = df[df["status"] != "CRASH"].dropna(subset=["turnover"])
ax.hist(valid["turnover"], bins=40, color="#3498db", alpha=0.85, edgecolor="black", linewidth=0.4)
ax.axvline(50.0, color="#c0392b", linestyle="--", linewidth=1.5, alpha=0.7, label="Turnover hard constraint")
ax.set_xlabel("Annualized turnover (one-sided)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Turnover Distribution Across Finished Runs", fontsize=13)
ax.grid(True, alpha=0.2)
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

## Walk-forward CV on kept rows

For each KEPT commit, check it out and run `uv run walkforward.py` to see per-fold Sharpes across 5 non-overlapping 2-year windows (2014–2023). A kept row whose fold Sharpes range from e.g. 0.2 → 1.8 is much more suspicious than one whose folds cluster 0.8 → 1.0 — same headline, very different signal quality.

The cell below prints the shell commands to run. It does **not** do the checkouts itself — walk-forward switches the working tree, which is disruptive to an in-progress experiment loop.

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
if kept.empty:
    print("No kept rows yet — nothing to walk-forward.")
else:
    print("# Run these at a shell, one kept commit at a time:")
    for _, row in kept.iterrows():
        sha = str(row["commit"]).strip()
        desc = str(row["description"]).strip()[:50]
        print(f"git checkout {sha} && uv run walkforward.py  # {desc}")
    print("\n# When done:")
    print("git checkout -")

## Deflated Sharpe (Bailey–López de Prado)

With N experiments, the expected max Sharpe under the null hypothesis (no real edge) is strictly positive and grows with N. A naive "point best > baseline point" comparison ignores this. The Deflated Sharpe Ratio (DSR) answers: *what is the probability that the best observed Sharpe is real, given we ran N trials and the observed cross-trial Sharpe dispersion?*

Uses a Gaussian approximation (skew=0, kurt=3) on the daily strategy returns — we only have summary stats in `results.tsv`, not the daily series. A finer estimate would re-run each kept strategy; for a quick gatekeeper, this is enough.

Rule of thumb: `DSR < 0.95` on your best kept row means the selection itself could be noise; investigate before deploying any capital against it (not that this repo deploys capital — but the logic transfers).

In [ ]:
from stats import deflated_sharpe, expected_max_sharpe_null

valid = df[df["status"] != "CRASH"].dropna(subset=["oos_sharpe"])
if valid.empty:
    print("No non-crashed runs yet.")
else:
    sharpes = valid["oos_sharpe"].to_numpy()
    n_trials = len(sharpes)
    best = float(sharpes.max())
    dsr = deflated_sharpe(best, sharpes)
    sigma = float(np.std(sharpes, ddof=1)) if n_trials >= 2 else float("nan")
    sr0 = expected_max_sharpe_null(sigma, n_trials)
    print(f"Trials N:                    {n_trials}")
    print(f"Cross-trial Sharpe sigma:    {sigma:.4f}")
    print(f"Expected max under null SR0: {sr0:.4f}")
    print(f"Best observed Sharpe:        {best:.4f}")
    print(f"Deflated Sharpe p-value:     {dsr:.4f}   (want > 0.95)")

## Parameter-sweep heuristic

Chains of kept rows that differ only by a number are the classic overfitting footprint. This cell normalizes each kept description (strips digits, collapses whitespace) and flags pairs that share the same "shape" — a soft signal for the human reviewer, not a hard rule.

In [ ]:
import re
from collections import defaultdict


def _shape(desc: str) -> str:
    s = str(desc).lower().strip()
    s = re.sub(r"\bthesis:\s*", "", s)
    # Replace any number (including decimals) with a single marker.
    s = re.sub(r"-?\d+(\.\d+)?", "N", s)
    s = re.sub(r"\s+", " ", s)
    return s


kept = df[df["status"] == "KEEP"].copy()
groups: dict[str, list[int]] = defaultdict(list)
for idx, row in kept.iterrows():
    groups[_shape(row["description"])].append(idx)

sweeps = {shape: idxs for shape, idxs in groups.items() if len(idxs) >= 2}
if not sweeps:
    print("No parameter-sweep chains detected among kept rows.")
else:
    print(f"Suspected parameter sweeps ({len(sweeps)}):\n")
    for shape, idxs in sweeps.items():
        print(f"  shape: '{shape}'")
        for i in idxs:
            sh = kept.loc[i, "oos_sharpe"]
            print(f"    #{i}  sharpe={sh:.4f}  {kept.loc[i, 'description']}")
        print()